# UN M7 manual runner — fixed-parameter model

Notebook per lanciare il modello M7 con parametri fissati manualmente.

Struttura:
1. cella parametri manuali;
2. core del modello;
3. funzioni di run/plot;
4. esecuzione.

Default: `capture_only` con bulk→dislocation capture attiva e \(D_v\) secondo la formula originale di Rizk 2025 (Eq. 4 + Table 2).


In [1]:
# ============================================================
# USER SETTINGS — modify only this cell for manual experiments
# ============================================================

# Output folder for CSV and PNG files
MANUAL_OUTPUT_DIR = "UN_M7_manual_fixed_params_results"
MANUAL_LABEL = "manual_Rizk2025_capture_literalDv"
MANUAL_SHOW_PLOTS = False

# Numerics. Use 1 h / 40 modes for final-quality plots; 6 h / 30 modes is faster.
MANUAL_DT_H = 6.0
MANUAL_N_MODES = 30

# Temperature and burnup grids
MANUAL_T_MIN = 900.0
MANUAL_T_MAX = 2600.0
MANUAL_T_STEP = 50.0
MANUAL_BURNUPS = [1.1, 1.3, 3.2]
MANUAL_GAS_PARTITION_T_MAX = 2600.0

# Physical constants / nominal baseline values used by the manual model.
GRAIN_RADIUS = 6.0e-6
XE_YIELD = 0.24
GAMMA_B = 1.11
OMEGA_FG = 8.5e-29
LATTICE_PARAMETER = 4.889e-10

FISSION_RATE_NOMINAL = 5.0e19
F_N_NOMINAL = 1.0e-6
K_D_NOMINAL = 5.0e5
RHO_D_NOMINAL = 3.0e13

# Physics family switches.
# capture_only = no phi in gas resolution, no nucleation mass coupling, capture ON.
MANUAL_USE_PHI_GAS_RESOLUTION = False
MANUAL_USE_NUCLEATION_MASS_COUPLING = False
MANUAL_USE_BULK_DISLOCATION_CAPTURE = True

# Plot set: same diagnostic plot list as capture_only outputs, without the pressure-ratio plot.
PLOT_CAPTURE_ONLY_STYLE_SET = True
# Extra optional plots for radius/concentration/pressure/gas partition at all burnups.
PLOT_EXTRA_ALL_BURNUPS = False

# Manual physical parameters.
# Baseline here is Rizk-like + coalescence + bulk→dislocation capture.
MANUAL_PARAMS = {
    "f_n": F_N_NOMINAL,
    "K_d": K_D_NOMINAL,
    "rho_d": RHO_D_NOMINAL,
    "fission_rate": FISSION_RATE_NOMINAL,

    # Global scale factors.
    "Dv_scale": 1.0,
    "Dg_scale": 1.0,
    "b_scale": 1.0,
    "gb_scale": 1.0,
    "gd_scale": 1.0,
    "coalescence_d_scale": 1.0,
    "capture_scale": 1.0,

    # Split scale factors used by the solver.
    "Dg_D1_scale": 1.0,
    "Dg_D3_scale": 1.0,
    "Dv_D1_scale": 1.0,
    "Dv_D2_scale": 1.0,
    "b_bulk_scale": 1.0,
    "b_dislocation_scale": 1.0,
    "gd_bubble_scale": 1.0,
    "gd_line_scale": 1.0,
    "gd_line_alpha": 1.0,
}

# Vacancy diffusivity option.
# "rizk2025_literal" uses Eq. (4) and Table 2 exactly as written in Rizk 2025:
#   Dv2 = sqrt(Fdot) * A20 * exp(-B21/kBT - B22/(kBT)^2)
MANUAL_DV_FORMULA = "rizk2025_literal"  # options: "rizk2025_literal", "v14_solver_signs_manual_A20", "v14_fig4_refit"
MANUAL_A20_VU = 1.32e-19
MANUAL_A20_VU_FIG4_REFIT = 4.6304523933553033e-29
MANUAL_B21_VU = -0.62
MANUAL_B22_VU = -0.04

# Dislocation-density mode.
# "constant" uses MANUAL_PARAMS["rho_d"].
# "rhoSat_RayBlank" uses the v14-style saturating Ray/Blank shape with a global scale.
MANUAL_RHO_MODE = "constant"  # options: "constant", "rhoSat_RayBlank"
MANUAL_RHO_SCALE = 1.0
MANUAL_RHO_FAB = RHO_D_NOMINAL

# Safety/diagnostic: clear run cache every time before producing plots.
MANUAL_CLEAR_CACHE_BEFORE_RUN = True


## Solver core

Questa cella contiene solo definizioni necessarie al run manuale: dati sperimentali, parametri, funzioni fisiche, solver e wrapper `run_model_point()`.

In [2]:
"""
UN M7 manual solver core.

This cell contains only the definitions needed to run fixed-parameter cases:
experimental points, ManualCase/UNParameters, physical helper functions,
solve_UN_M7(), and run_model_point().
"""

from __future__ import annotations

import csv
import math
import os
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Sequence, Tuple

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

# Family switches are set from the first cell by apply_manual_switches().
USE_PHI_GAS_RESOLUTION = False
USE_NUCLEATION_MASS_COUPLING = False
USE_BULK_DISLOCATION_CAPTURE = True
MODEL_FAMILY = "manual"


EXP_SWELLING_T = [
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1127.0, "swelling": 0.68},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1228.0, "swelling": 0.59},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1312.0, "swelling": 0.43},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1402.0, "swelling": 0.58},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1485.0, "swelling": 1.22},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1549.0, "swelling": 1.84},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1598.0, "swelling": 1.66},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1632.0, "swelling": 2.13},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1669.0, "swelling": 3.60},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1685.0, "swelling": 2.72},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 899.0,  "swelling": 0.63},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1154.0, "swelling": 1.17},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1228.0, "swelling": 1.08},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1325.0, "swelling": 1.28},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1435.0, "swelling": 1.32},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1514.0, "swelling": 2.10},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1570.0, "swelling": 2.72},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1608.0, "swelling": 2.91},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1635.0, "swelling": 3.28},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1656.0, "swelling": 3.75},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1044.0, "swelling": 1.11},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1220.0, "swelling": 1.22},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1377.0, "swelling": 1.33},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1534.0, "swelling": 2.83},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1595.0, "swelling": 3.53},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1639.0, "swelling": 3.86},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1661.0, "swelling": 2.45},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1709.0, "swelling": 2.93},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1724.0, "swelling": 3.15},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 984.0,  "swelling": 0.72},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1056.0, "swelling": 1.06},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1126.0, "swelling": 1.26},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1247.0, "swelling": 1.58},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1343.0, "swelling": 1.79},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1420.0, "swelling": 2.08},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1459.0, "swelling": 2.40},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1511.0, "swelling": 2.83},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1557.0, "swelling": 3.31},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1590.0, "swelling": 3.75},
]

EXP_SWELLING_BURNUP_1600 = [
    {"burnup": 1.12, "swelling": 1.64, "series": "approx 100 kW/m"},
    {"burnup": 1.11, "swelling": 2.90, "series": "approx 119 kW/m"},
    {"burnup": 1.31, "swelling": 3.51, "series": "measurement"},
    {"burnup": 3.18, "swelling": 3.72, "series": "measurement"},
]

EXP_ND_T_13 = [
    {"T": 1153.0, "N": 5.22e19}, {"T": 1202.0, "N": 3.69e19},
    {"T": 1227.0, "N": 3.00e19}, {"T": 1235.0, "N": 3.57e19},
    {"T": 1248.0, "N": 2.44e19}, {"T": 1330.0, "N": 2.90e19},
    {"T": 1338.0, "N": 4.10e19}, {"T": 1377.0, "N": 2.71e19},
    {"T": 1408.0, "N": 1.67e19}, {"T": 1427.0, "N": 1.85e19},
    {"T": 1493.0, "N": 3.00e19}, {"T": 1507.0, "N": 2.44e19},
    {"T": 1524.0, "N": 1.56e19}, {"T": 1538.0, "N": 7.54e18},
    {"T": 1555.0, "N": 7.80e18}, {"T": 1561.0, "N": 1.92e19},
    {"T": 1599.0, "N": 5.34e18}, {"T": 1622.0, "N": 7.80e18},
    {"T": 1628.0, "N": 1.73e19}, {"T": 1649.0, "N": 5.34e18},
    {"T": 1656.0, "N": 3.65e18}, {"T": 1656.0, "N": 2.41e18},
    {"T": 1669.0, "N": 1.96e18}, {"T": 1682.0, "N": 2.44e19},
    {"T": 1685.0, "N": 1.40e19}, {"T": 1723.0, "N": 7.70e17},
    {"T": 1742.0, "N": 1.02e18}, {"T": 1739.0, "N": 1.65e18},
]

EXP_RD_T_13 = [
    {"T": 1045.0, "R_nm": 54.06}, {"T": 1219.0, "R_nm": 60.37},
    {"T": 1374.0, "R_nm": 69.58}, {"T": 1535.0, "R_nm": 104.85},
    {"T": 1594.0, "R_nm": 120.83}, {"T": 1641.0, "R_nm": 143.73},
    {"T": 1663.0, "R_nm": 122.76}, {"T": 1710.0, "R_nm": 157.99},
    {"T": 1725.0, "R_nm": 173.67},
]

# ============================================================
# MODEL PARAMETERS
# ============================================================

@dataclass(frozen=True)
class ManualCase:
    label: str
    f_n: float
    K_d: float
    rho_d: float
    fission_rate: float

    # Global scale factors
    Dv_scale: float = 1.0
    Dg_scale: float = 1.0
    b_scale: float = 1.0
    gb_scale: float = 1.0
    gd_scale: float = 1.0
    coalescence_d_scale: float = 1.0
    capture_scale: float = 1.0

    # Split gas diffusion: D_g = Dg_scale * (Dg_D1_scale*D1 + Dg_D3_scale*D3)
    Dg_D1_scale: float = 1.0
    Dg_D3_scale: float = 1.0

    # Split vacancy diffusion: D_v = Dv_scale * (Dv_D1_scale*D1 + Dv_D2_scale*D2)
    Dv_D1_scale: float = 1.0
    Dv_D2_scale: float = 1.0

    # Split re-solution
    b_bulk_scale: float = 1.0
    b_dislocation_scale: float = 1.0

    # Split dislocation trapping
    gd_bubble_scale: float = 1.0
    gd_line_scale: float = 1.0
    # alpha=1: free-dislocation correction rho_d - 2 R_d N_d
    # alpha=0: full line sink rho_d
    gd_line_alpha: float = 1.0

    # Diagnostic only: Xe D2 is computed but not included in D_g
    D2_xe_scale: float = 1.0

@dataclass
class UNParameters:
    temperature: float = 1600.0
    fission_rate: float = FISSION_RATE_NOMINAL
    grain_radius: float = GRAIN_RADIUS
    target_burnup_percent_fima: Optional[float] = None
    final_time: float = 24.0 * 3600.0
    dt: float = 3600.0
    n_modes: int = 40

    xe_yield: float = XE_YIELD
    precursor_factor: float = 1.0

    # Xe diffusivity: D1 + D2 + D3 (Rizk Table 2).
    D10: float = 1.56e-3
    Q1: float = 4.94
    A20_xe: float = 1.21e-67
    B21_xe: float = 25.87
    B22_xe: float = -1.49
    B23_xe: float = 0.0
    A30: float = 1.85e-39
    D2_xe_scale: float = 1.0  # diagnostic only; Xe D2 is not used in D_g
    Dg_scale: float = 1.0
    Dg_D1_scale: float = 1.0
    Dg_D3_scale: float = 1.0

    kB_eV: float = 8.617333262e-5
    kB_J: float = 1.380649e-23

    # Vacancy diffusivity parameters for U vacancies (Rizk 2025 Table 2 coefficients).
    D10_vU: float = 1.35e-2
    Q1_vU: float = 5.66
    B21_vU_refit: float = -0.62
    B22_vU_refit: float = -0.04
    A20_vU_fig4_refit: float = 4.6304523933553033e-29
    Dv_scale: float = 1.0
    Dv_D1_scale: float = 1.0
    Dv_D2_scale: float = 1.0

    radius_in_lattice: float = 0.21e-9
    omega_fg: float = OMEGA_FG
    lattice_parameter: float = LATTICE_PARAMETER
    gamma_b: float = GAMMA_B
    hydrostatic_stress: float = 0.0
    min_radius_for_pressure: float = 1.0e-15

    f_n: float = F_N_NOMINAL
    rho_d: float = RHO_D_NOMINAL
    K_d: float = K_D_NOMINAL
    r_d: float = 3.46e-10
    Z_d: float = 5.0

    Dg_extra_scale: float = 1.0
    gb_scale: float = 1.0
    gd_scale: float = 1.0
    b_scale: float = 1.0
    b_bulk_scale: float = 1.0
    b_dislocation_scale: float = 1.0
    gd_bubble_scale: float = 1.0
    gd_line_scale: float = 1.0
    gd_line_alpha: float = 1.0
    coalescence_d_scale: float = 1.0
    capture_scale: float = 1.0

    R_b: float = 0.0
    N_b: float = 0.0
    R_d: float = 0.0
    N_d: Optional[float] = None
    c0: float = 0.0
    mb0: float = 0.0
    md0: float = 0.0
    nvb0: Optional[float] = None
    nvd0: Optional[float] = None

    bulk_seed_radius_nm: float = 0.0
    vacancy_absorption_only: bool = True
    update_bulk_vacancies: bool = True
    min_number_density: float = 0.0
    min_volume: float = 0.0

    def __post_init__(self):
        if self.N_d is None:
            self.N_d = self.K_d * self.rho_d
        if self.target_burnup_percent_fima is not None:
            self.final_time = burnup_percent_to_time(
                self.target_burnup_percent_fima,
                self.fission_rate,
                self.lattice_parameter,
            )

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def omega_matrix(p: UNParameters) -> float:
    return p.lattice_parameter**3 / 4.0


def uranium_atom_density_from_lattice(lattice_parameter: float) -> float:
    return 4.0 / lattice_parameter**3


def burnup_percent_to_time(burnup_percent_fima: float, fission_rate: float, lattice_parameter: float) -> float:
    if fission_rate <= 0.0:
        raise ValueError("fission_rate must be positive")
    return (burnup_percent_fima / 100.0) * uranium_atom_density_from_lattice(lattice_parameter) / fission_rate


def time_to_burnup_percent(time: float, fission_rate: float, lattice_parameter: float) -> float:
    return 100.0 * fission_rate * time / uranium_atom_density_from_lattice(lattice_parameter)


def sphere_volume(R: float) -> float:
    return 0.0 if R <= 0.0 else (4.0 / 3.0) * math.pi * R**3


def radius_from_volume(V: float) -> float:
    return 0.0 if V <= 0.0 else (3.0 * V / (4.0 * math.pi)) ** (1.0 / 3.0)


def xe_diffusivity_UN(p: UNParameters):
    T = p.temperature
    kBT = p.kB_eV * T
    D1 = p.D10 * math.exp(-p.Q1 / kBT)
    # Irradiation-enhanced Xe diffusivity. Rizk notes it is usually small,
    # but it is kept explicitly here instead of being set to zero.
    try:
        expo = (
            -p.B21_xe / kBT
            -p.B22_xe / (kBT**2)
            -p.B23_xe / (kBT**3)
        )
        # avoid numerical overflow if trial ranges are later modified
        expo = max(min(expo, 700.0), -745.0)
        D2 = math.sqrt(p.fission_rate) * p.A20_xe * math.exp(expo)
    except OverflowError:
        D2 = math.inf
    D3 = p.A30 * p.fission_rate

    # for Xe compared with D1 + D3.  D2 is still returned as a diagnostic.
    Dg_unscaled = p.Dg_D1_scale * D1 + p.Dg_D3_scale * D3
    Dg = Dg_unscaled * p.Dg_scale * p.precursor_factor * p.Dg_extra_scale
    return Dg, {
        "D1_Xe": D1,
        "D2_Xe": D2,
        "D2_Xe_scaled": 0.0,
        "D3_Xe": D3,
        "Dg_D1_scaled": p.Dg_D1_scale * D1,
        "Dg_D3_scaled": p.Dg_D3_scale * D3,
        "D2_Xe_over_Dg_unscaled": (D2 / Dg_unscaled) if Dg_unscaled > 0 and math.isfinite(Dg_unscaled) else math.nan,
        "Dg": Dg,
    }


def vacancy_diffusivity_UN(p: UNParameters):
    T = p.temperature
    kBT = p.kB_eV * T
    D1 = p.D10_vU * math.exp(-p.Q1_vU / kBT)
    D2 = math.sqrt(p.fission_rate) * p.A20_vU_fig4_refit * math.exp(
        p.B21_vU_refit / kBT + p.B22_vU_refit / (kBT**2)
    )
    Dv_unscaled = p.Dv_D1_scale * D1 + p.Dv_D2_scale * D2
    Dv = Dv_unscaled * p.Dv_scale
    return Dv, {
        "Dv1": D1,
        "Dv2": D2,
        "Dv1_scaled": p.Dv_D1_scale * D1,
        "Dv2_scaled": p.Dv_D2_scale * D2,
        "Dv": Dv,
    }


def b0_resolution(R: float) -> float:
    R = max(R, 1.0e-15)
    return 1.0e-25 * (2.64 - 2.02 * math.exp(-2.61e-9 / R))


def resolution_rates_UN(p: UNParameters, R_b: float, R_d: float):
    b_b = p.fission_rate * b0_resolution(R_b + p.radius_in_lattice) * p.b_scale * p.b_bulk_scale
    b_d = p.fission_rate * b0_resolution(R_d + p.radius_in_lattice) * p.b_scale * p.b_dislocation_scale
    return b_b, b_d


def trapping_rates_UN(p: UNParameters, Dg: float, R_b: float, N_b: float, R_d: float, N_d: float):
    Rb_eff = R_b + p.radius_in_lattice
    Rd_eff = R_d + p.radius_in_lattice
    g_b_unscaled = 0.0 if N_b <= 0.0 else 4.0 * math.pi * Dg * Rb_eff * N_b

    Gamma_d = 1.0 / math.sqrt(math.pi * p.rho_d)
    den = math.log(Gamma_d / (p.Z_d * p.r_d)) - 3.0 / 5.0
    if den <= 0.0:
        raise ValueError(f"Invalid dislocation sink denominator: {den:g}")

    # gd_line_alpha=1 reproduces the previous correction rho_d - 2 R_d N_d.
    # gd_line_alpha=0 gives a Barani-like full line sink rho_d.
    free_dislocation = max(p.rho_d - p.gd_line_alpha * 2.0 * R_d * N_d, 0.0)
    term_bubbles = 4.0 * math.pi * Dg * Rd_eff * N_d
    term_dislocation = (2.0 * math.pi * Dg / den) * free_dislocation
    g_d_unscaled = p.gd_bubble_scale * term_bubbles + p.gd_line_scale * term_dislocation

    g_b = p.gb_scale * g_b_unscaled
    g_d = p.gd_scale * g_d_unscaled

    return g_b, g_d, {
        "Gamma_d": Gamma_d,
        "den": den,
        "free_dislocation": free_dislocation,
        "term_bubbles": term_bubbles,
        "term_dislocation": term_dislocation,
        "term_bubbles_scaled": p.gd_bubble_scale * term_bubbles,
        "term_dislocation_scaled": p.gd_line_scale * term_dislocation,
        "g_b_unscaled": g_b_unscaled,
        "g_d_unscaled": g_d_unscaled,
    }


def beta_production(p: UNParameters) -> float:
    return p.xe_yield * p.fission_rate


def nucleation_rate_bulk(p: UNParameters, Dg: float, c: float) -> float:
    return 8.0 * math.pi * p.f_n * Dg * p.omega_fg ** (1.0 / 3.0) * max(c, 0.0) ** 2


def phi_population(m_gas: float, N: float) -> float:
    if N <= 0.0 or m_gas <= 0.0:
        return 0.0
    atoms_per_bubble = m_gas / N
    if atoms_per_bubble <= 1.0:
        return 0.0
    return 1.0 / (atoms_per_bubble - 1.0)


def coalescence_lambda(Vd: float, Nd: float) -> float:
    xi = max(0.0, min(Vd * Nd, 0.999999))
    return (2.0 - xi) / (2.0 * (1.0 - xi) ** 3)


def pressure_internal(p: UNParameters, m_gas: float, n_vac: float) -> float:
    if m_gas <= 0.0:
        return 0.0
    if n_vac <= 0.0:
        return math.inf
    denom = n_vac * omega_matrix(p)
    return math.inf if denom <= 0.0 else p.kB_J * p.temperature * m_gas / denom


def pressure_equilibrium(p: UNParameters, R: float) -> float:
    return 2.0 * p.gamma_b / max(R, p.min_radius_for_pressure) - p.hydrostatic_stress


def gas_only_radius_for_population(p: UNParameters, m_gas: float, N: float) -> float:
    if m_gas <= 0.0 or N <= 0.0:
        return 0.0
    return radius_from_volume(p.omega_fg * m_gas / N)


def radius_for_vacancy_update(p: UNParameters, R_old: float, N: float, m_gas: float) -> float:
    if R_old > 0.0:
        return R_old
    return gas_only_radius_for_population(p, m_gas, N)


def wigner_seitz_delta(N: float) -> float:
    return (3.0 / (4.0 * math.pi * max(N, 1.0))) ** (1.0 / 3.0)


def zeta_geometry(R: float, N: float) -> float:
    """Geometric factor for vacancy absorption, overflow-safe but v14-equivalent.

    The original v14 implementation was:
        psi = max(R / delta, 1e-12)
        den = -psi**6 + 5*psi**2 - 9*psi + 5
        den = max(den, 1e-30)
        zeta = max(10*psi*(1+psi**3)/den, 1e-30)

    We keep that behavior.  The only change is an overflow guard for extreme
    nonphysical single-size states where psi becomes so large that psi**6
    overflows.  In that limit the original formula would have used den=1e-30
    and an enormous zeta, which makes the vacancy update rate tend to zero.
    Returning a huge finite zeta is therefore the safest v14-preserving fix.
    """
    delta = wigner_seitz_delta(N)
    psi = max(R / delta, 1.0e-12) if delta > 0.0 else math.inf

    if (not math.isfinite(psi)) or psi > 1.0e50:
        return 1.0e300

    try:
        den = -psi**6 + 5.0 * psi**2 - 9.0 * psi + 5.0
        den = max(den, 1.0e-30)
        zeta = 10.0 * psi * (1.0 + psi**3) / den
    except OverflowError:
        return 1.0e300

    if not math.isfinite(zeta):
        return 1.0e300
    return max(zeta, 1.0e-30)


def vacancy_concentration_implicit_step(p: UNParameters, Dv: float, R: float, N: float, m_gas: float, n_old: float, dt: float):
    if N <= 0.0 or m_gas <= 0.0:
        return n_old, 0.0
    R_update = radius_for_vacancy_update(p, R, N, m_gas)
    if R_update <= 0.0:
        return n_old, 0.0

    p_eq = 2.0 * p.gamma_b / R_update - p.hydrostatic_stress
    p_int_old = pressure_internal(p, m_gas, n_old)

    if p.vacancy_absorption_only and p_int_old <= p_eq:
        return n_old, 0.0

    delta = wigner_seitz_delta(N)
    zeta = zeta_geometry(R_update, N)
    A = 2.0 * math.pi * Dv * delta * N / (p.kB_J * p.temperature * zeta)
    C = p.kB_J * p.temperature * m_gas / omega_matrix(p)
    B = n_old - dt * A * p_eq
    disc = B * B + 4.0 * dt * A * C

    if disc < 0.0:
        raise ValueError(f"Negative discriminant in vacancy step: {disc:g}")

    sqrt_disc = math.sqrt(disc)
    if B >= 0.0:
        n_new = 0.5 * (B + sqrt_disc)
    else:
        denom = sqrt_disc - B
        n_new = 0.0 if denom <= 0.0 else (2.0 * dt * A * C) / denom

    if p.vacancy_absorption_only:
        n_new = max(n_new, n_old)

    return n_new, (n_new - n_old) / dt


def initialize_vacancy_concentration(p: UNParameters, N: float, R: float, m_gas: float) -> float:
    if N <= 0.0 or R <= 0.0:
        return 0.0
    vacancy_volume = max(N * sphere_volume(R) - p.omega_fg * m_gas, 0.0)
    return vacancy_volume / omega_matrix(p)


def initialize_modes_from_average(average: float, n_modes: int, n_iter: int = 20):
    modes = [0.0 for _ in range(n_modes)]
    projection_coeff = -math.sqrt(8.0 / math.pi)
    remainder = average
    for _ in range(n_iter):
        reconstructed = 0.0
        for i in range(n_modes):
            n = i + 1
            n_coeff = (-1.0) ** n / n
            modes[i] += projection_coeff * n_coeff * remainder
            reconstructed += projection_coeff * n_coeff * modes[i] * 3.0 / (4.0 * math.pi)
        remainder = average - reconstructed
    return modes


def reconstruct_average(modes: Sequence[float]) -> float:
    projection_coeff = -2.0 * math.sqrt(2.0 / math.pi)
    average = 0.0
    for i, value in enumerate(modes):
        n = i + 1
        n_coeff = (-1.0) ** n / n
        average += projection_coeff * n_coeff * value / ((4.0 / 3.0) * math.pi)
    return average


def det3(A):
    return (
        A[0][0] * (A[1][1] * A[2][2] - A[1][2] * A[2][1])
        - A[0][1] * (A[1][0] * A[2][2] - A[1][2] * A[2][0])
        + A[0][2] * (A[1][0] * A[2][1] - A[1][1] * A[2][0])
    )


def solve3x3_cramer(A, b):
    detA = det3(A)
    if abs(detA) < 1.0e-300:
        raise ZeroDivisionError("Singular 3x3 system")
    Ax = [[b[i], A[i][1], A[i][2]] for i in range(3)]
    Ay = [[A[i][0], b[i], A[i][2]] for i in range(3)]
    Az = [[A[i][0], A[i][1], b[i]] for i in range(3)]
    return [det3(Ax) / detA, det3(Ay) / detA, det3(Az) / detA]


def sciantix_3x3_exchange_step(
    modes_c,
    modes_mb,
    modes_md,
    Dg: float,
    R_grain: float,
    source_c: float,
    source_mb: float,
    source_md: float,
    g_b: float,
    g_d: float,
    b_b_gas: float,
    b_d_gas: float,
    dt: float,
):
    projection_coeff = -2.0 * math.sqrt(2.0 / math.pi)
    diffusion_rate_coeff = math.pi**2 * Dg / R_grain**2

    for i in range(len(modes_c)):
        n = i + 1
        n_coeff = (-1.0) ** n / n
        diffusion_rate = diffusion_rate_coeff * n**2

        src_c = projection_coeff * source_c * n_coeff
        src_mb = projection_coeff * source_mb * n_coeff
        src_md = projection_coeff * source_md * n_coeff

        A = [
            [1.0 + (diffusion_rate + g_b + g_d) * dt, -b_b_gas * dt, -b_d_gas * dt],
            [-g_b * dt, 1.0 + b_b_gas * dt, 0.0],
            [-g_d * dt, 0.0, 1.0 + b_d_gas * dt],
        ]
        rhs = [
            modes_c[i] + src_c * dt,
            modes_mb[i] + src_mb * dt,
            modes_md[i] + src_md * dt,
        ]
        modes_c[i], modes_mb[i], modes_md[i] = solve3x3_cramer(A, rhs)

    return reconstruct_average(modes_c), reconstruct_average(modes_mb), reconstruct_average(modes_md)


def reset_modes_to_averages(c: float, mb: float, md: float, n_modes: int):
    return (
        initialize_modes_from_average(max(c, 0.0), n_modes),
        initialize_modes_from_average(max(mb, 0.0), n_modes),
        initialize_modes_from_average(max(md, 0.0), n_modes),
    )

# ============================================================
# SOLVER M7
# ============================================================

def solve_UN_M7(p: UNParameters, keep_history: bool = True):
    modes_c = initialize_modes_from_average(p.c0, p.n_modes)
    modes_mb = initialize_modes_from_average(p.mb0, p.n_modes)
    modes_md = initialize_modes_from_average(p.md0, p.n_modes)

    R_b = p.R_b
    R_d = p.R_d
    N_b = p.N_b
    N_d = p.N_d
    V_b = sphere_volume(R_b)
    V_d = sphere_volume(R_d)

    nvb = initialize_vacancy_concentration(p, N_b, R_b, p.mb0) if p.nvb0 is None else p.nvb0
    nvd = initialize_vacancy_concentration(p, N_d, R_d, p.md0) if p.nvd0 is None else p.nvd0

    beta = beta_production(p)
    initial_gas = p.c0 + p.mb0 + p.md0
    generated = 0.0
    q_gb = 0.0
    retained = initial_gas
    t = 0.0

    capture_fraction_sum = 0.0
    capture_raw_sum = 0.0
    capture_bubbles_cumulative = 0.0
    max_f_cap_step = 0.0

    hist_keys = [
        "time", "burnup_percent_fima", "c", "mb", "md", "Nb", "Nd", "Vb", "Vd", "Rb", "Rd",
        "nvb", "nvd", "generated", "retained", "q_gb", "swelling_b", "swelling_d", "swelling_ig",
        "p_b", "p_d", "p_b_eq", "p_d_eq", "lambda_d", "nu_b", "phi_b", "phi_d",
        "f_cap_step", "cap_raw_step", "capture_fraction_sum", "capture_raw_sum",
        "capture_bubbles_cumulative", "max_f_cap_step",
        "matrix_gas_percent", "bulk_gas_percent", "dislocation_gas_percent", "qgb_gas_percent",
    ]
    hist = {key: [] for key in hist_keys}

    def append_state(nu_b=0.0, phi_b=0.0, phi_d=0.0, lambda_d=0.0, fcap=0.0, capraw=0.0):
        if not keep_history:
            return
        c_av = reconstruct_average(modes_c)
        mb_av = reconstruct_average(modes_mb)
        md_av = reconstruct_average(modes_md)
        p_b = pressure_internal(p, mb_av, nvb)
        p_d = pressure_internal(p, md_av, nvd)
        p_b_eq = pressure_equilibrium(p, R_b)
        p_d_eq = pressure_equilibrium(p, R_d)

        hist["time"].append(t)
        hist["burnup_percent_fima"].append(time_to_burnup_percent(t, p.fission_rate, p.lattice_parameter))
        hist["c"].append(c_av)
        hist["mb"].append(mb_av)
        hist["md"].append(md_av)
        hist["Nb"].append(N_b)
        hist["Nd"].append(N_d)
        hist["Vb"].append(V_b)
        hist["Vd"].append(V_d)
        hist["Rb"].append(R_b)
        hist["Rd"].append(R_d)
        hist["nvb"].append(nvb)
        hist["nvd"].append(nvd)
        hist["generated"].append(generated)
        hist["retained"].append(retained)
        hist["q_gb"].append(q_gb)
        hist["swelling_b"].append(N_b * V_b)
        hist["swelling_d"].append(N_d * V_d)
        hist["swelling_ig"].append(N_b * V_b + N_d * V_d)
        hist["p_b"].append(p_b)
        hist["p_d"].append(p_d)
        hist["p_b_eq"].append(p_b_eq)
        hist["p_d_eq"].append(p_d_eq)
        hist["lambda_d"].append(lambda_d)
        hist["nu_b"].append(nu_b)
        hist["phi_b"].append(phi_b)
        hist["phi_d"].append(phi_d)
        hist["f_cap_step"].append(fcap)
        hist["cap_raw_step"].append(capraw)
        hist["capture_fraction_sum"].append(capture_fraction_sum)
        hist["capture_raw_sum"].append(capture_raw_sum)
        hist["capture_bubbles_cumulative"].append(capture_bubbles_cumulative)
        hist["max_f_cap_step"].append(max_f_cap_step)
        hist["matrix_gas_percent"].append(100.0 * c_av / generated if generated > 0.0 else 0.0)
        hist["bulk_gas_percent"].append(100.0 * mb_av / generated if generated > 0.0 else 0.0)
        hist["dislocation_gas_percent"].append(100.0 * md_av / generated if generated > 0.0 else 0.0)
        hist["qgb_gas_percent"].append(100.0 * q_gb / generated if generated > 0.0 else 0.0)

    append_state()
    last_rates = {}
    n_steps = int(math.ceil(p.final_time / p.dt))

    for _ in range(n_steps):
        dt = min(p.dt, p.final_time - t)
        if dt <= 0.0:
            break

        c_old = reconstruct_average(modes_c)
        mb_old = reconstruct_average(modes_mb)
        md_old = reconstruct_average(modes_md)

        Nb_old = N_b
        Nd_old = N_d
        Vd_old = V_d
        Rb_old = R_b
        Rd_old = R_d

        Dg, D_parts = xe_diffusivity_UN(p)
        Dv, Dv_parts = vacancy_diffusivity_UN(p)
        b_b, b_d = resolution_rates_UN(p, R_b, R_d)
        g_b, g_d, trapping_parts = trapping_rates_UN(p, Dg, R_b, Nb_old, R_d, Nd_old)

        nu_b = nucleation_rate_bulk(p, Dg, c_old)
        phi_b = phi_population(mb_old, Nb_old)
        phi_d = phi_population(md_old, Nd_old)

        if USE_PHI_GAS_RESOLUTION:
            b_b_gas = b_b * phi_b
            b_d_gas = b_d * phi_d
        else:
            b_b_gas = b_b
            b_d_gas = b_d

        # N_b destruction uses original b_b * phi_b term.
        N_b = (Nb_old + dt * nu_b) / (1.0 + dt * b_b * phi_b)
        N_b = max(N_b, p.min_number_density)

        # Optional M7 gas mass coupling of bulk nucleation.
        if USE_NUCLEATION_MASS_COUPLING:
            source_c = beta - 2.0 * nu_b
            source_mb = 2.0 * nu_b
        else:
            source_c = beta
            source_mb = 0.0
        source_md = 0.0

        c_new, mb_new, md_new = sciantix_3x3_exchange_step(
            modes_c, modes_mb, modes_md,
            Dg, p.grain_radius,
            source_c, source_mb, source_md,
            g_b, g_d,
            b_b_gas, b_d_gas,
            dt,
        )

        if c_new < 0.0 or mb_new < 0.0 or md_new < 0.0:
            c_new = max(c_new, 0.0)
            mb_new = max(mb_new, 0.0)
            md_new = max(md_new, 0.0)
            modes_c, modes_mb, modes_md = reset_modes_to_averages(c_new, mb_new, md_new, p.n_modes)

        dmb_dt = (mb_new - mb_old) / dt
        dmd_dt = (md_new - md_old) / dt

        if p.update_bulk_vacancies:
            nvb, dnvb_dt = vacancy_concentration_implicit_step(p, Dv, R_b, N_b, mb_new, nvb, dt)
        else:
            dnvb_dt = 0.0
        nvd, dnvd_dt = vacancy_concentration_implicit_step(p, Dv, R_d, Nd_old, md_new, nvd, dt)

        if N_b > 0.0:
            V_b_growth = V_b + dt * (p.omega_fg / N_b * dmb_dt + omega_matrix(p) / N_b * dnvb_dt)
            V_b_growth = max(V_b_growth, p.min_volume)
        else:
            V_b_growth = 0.0

        if Nd_old > 0.0:
            dVd_growth_dt = p.omega_fg / Nd_old * dmd_dt + omega_matrix(p) / Nd_old * dnvd_dt
            V_d_growth = max(V_d + dt * dVd_growth_dt, p.min_volume)
        else:
            dVd_growth_dt = 0.0
            V_d_growth = 0.0

        lambda_d = coalescence_lambda(Vd_old, Nd_old)
        dVd_positive = max(V_d_growth - Vd_old, 0.0)
        if dVd_positive > 0.0 and Nd_old > 0.0:
            denominator = 1.0 + p.coalescence_d_scale * 4.0 * lambda_d * Nd_old * dVd_positive
            N_d = Nd_old / denominator
        else:
            N_d = Nd_old
        N_d = max(N_d, p.min_number_density)

        # Reconstruct volumes after gas/vacancy update and dislocation coalescence.
        V_b = (p.omega_fg * max(mb_new, 0.0) + omega_matrix(p) * nvb) / N_b if N_b > 0.0 else 0.0
        V_d = (p.omega_fg * max(md_new, 0.0) + omega_matrix(p) * nvd) / N_d if N_d > 0.0 else 0.0
        V_b = max(V_b, p.min_volume)
        V_d = max(V_d, p.min_volume)
        R_b = radius_from_volume(V_b)
        R_d = radius_from_volume(V_d)

        # Barani uses dV*_d = 4*pi*(R_d + R_b)^2*dR_d.
        # The previous implementation used Delta[(R_d+R_b)^3], which also counted Delta R_b.
        # Here only the expansion of the dislocation bubble sweeps new capture volume.
        delta_Rd_cap = max(R_d - Rd_old, 0.0)
        delta_Vcap = 4.0 * math.pi * (Rd_old + Rb_old) ** 2 * delta_Rd_cap
        if USE_BULK_DISLOCATION_CAPTURE:
            cap_raw_step = p.capture_scale * N_d * delta_Vcap
            f_cap = max(0.0, min(cap_raw_step, 1.0))
        else:
            cap_raw_step = 0.0
            f_cap = 0.0

        capture_raw_sum += cap_raw_step
        capture_fraction_sum += f_cap
        max_f_cap_step = max(max_f_cap_step, f_cap)

        if f_cap > 0.0 and N_b > 0.0:
            mb_before = max(mb_new, 0.0)
            nvb_before = max(nvb, 0.0)
            captured_bubbles = f_cap * N_b

            mb_new = (1.0 - f_cap) * mb_before
            md_new = max(md_new, 0.0) + f_cap * mb_before
            nvb = (1.0 - f_cap) * nvb_before
            nvd = max(nvd, 0.0) + f_cap * nvb_before
            N_b = (1.0 - f_cap) * N_b

            capture_bubbles_cumulative += captured_bubbles

            modes_c, modes_mb, modes_md = reset_modes_to_averages(c_new, mb_new, md_new, p.n_modes)

            V_b = (p.omega_fg * max(mb_new, 0.0) + omega_matrix(p) * nvb) / N_b if N_b > 0.0 else 0.0
            V_d = (p.omega_fg * max(md_new, 0.0) + omega_matrix(p) * nvd) / N_d if N_d > 0.0 else 0.0
            V_b = max(V_b, p.min_volume)
            V_d = max(V_d, p.min_volume)
            R_b = radius_from_volume(V_b)
            R_d = radius_from_volume(V_d)

        generated += beta * dt
        retained = max(c_new, 0.0) + max(mb_new, 0.0) + max(md_new, 0.0)
        q_gb = max(initial_gas + generated - retained, 0.0)
        t += dt

        last_rates = {
            "Dg": Dg, "Dv": Dv, "beta": beta,
            "g_b": g_b, "g_d": g_d,
            "b_b": b_b, "b_d": b_d,
            "b_b_gas": b_b_gas, "b_d_gas": b_d_gas,
            "nu_b": nu_b, "phi_b": phi_b, "phi_d": phi_d,
            "lambda_d": lambda_d,
            "dVd_growth_dt": dVd_growth_dt,
            "dnvb_dt": dnvb_dt, "dnvd_dt": dnvd_dt,
            "f_cap_step": f_cap,
            "cap_raw_step": cap_raw_step,
            "capture_fraction_sum": capture_fraction_sum,
            "capture_raw_sum": capture_raw_sum,
            "capture_bubbles_cumulative": capture_bubbles_cumulative,
            "max_f_cap_step": max_f_cap_step,
            **D_parts, **Dv_parts, **trapping_parts,
        }

        append_state(nu_b=nu_b, phi_b=phi_b, phi_d=phi_d, lambda_d=lambda_d, fcap=f_cap, capraw=cap_raw_step)

    if not keep_history:
        c_av = reconstruct_average(modes_c)
        mb_av = reconstruct_average(modes_mb)
        md_av = reconstruct_average(modes_md)
        p_b = pressure_internal(p, mb_av, nvb)
        p_d = pressure_internal(p, md_av, nvd)
        p_b_eq = pressure_equilibrium(p, R_b)
        p_d_eq = pressure_equilibrium(p, R_d)
        hist = {
            "time": [t],
            "burnup_percent_fima": [time_to_burnup_percent(t, p.fission_rate, p.lattice_parameter)],
            "c": [c_av], "mb": [mb_av], "md": [md_av],
            "Nb": [N_b], "Nd": [N_d],
            "Vb": [V_b], "Vd": [V_d],
            "Rb": [R_b], "Rd": [R_d],
            "nvb": [nvb], "nvd": [nvd],
            "generated": [generated],
            "retained": [retained],
            "q_gb": [q_gb],
            "swelling_b": [N_b * V_b],
            "swelling_d": [N_d * V_d],
            "swelling_ig": [N_b * V_b + N_d * V_d],
            "p_b": [p_b], "p_d": [p_d],
            "p_b_eq": [p_b_eq], "p_d_eq": [p_d_eq],
            "lambda_d": [last_rates.get("lambda_d", 0.0)],
            "nu_b": [last_rates.get("nu_b", 0.0)],
            "phi_b": [last_rates.get("phi_b", 0.0)],
            "phi_d": [last_rates.get("phi_d", 0.0)],
            "f_cap_step": [last_rates.get("f_cap_step", 0.0)],
            "cap_raw_step": [last_rates.get("cap_raw_step", 0.0)],
            "capture_fraction_sum": [capture_fraction_sum],
            "capture_raw_sum": [capture_raw_sum],
            "capture_bubbles_cumulative": [capture_bubbles_cumulative],
            "max_f_cap_step": [max_f_cap_step],
            "matrix_gas_percent": [100.0 * c_av / generated if generated > 0.0 else 0.0],
            "bulk_gas_percent": [100.0 * mb_av / generated if generated > 0.0 else 0.0],
            "dislocation_gas_percent": [100.0 * md_av / generated if generated > 0.0 else 0.0],
            "qgb_gas_percent": [100.0 * q_gb / generated if generated > 0.0 else 0.0],
        }

    return hist, last_rates

# ============================================================
# RUN WRAPPER
# ============================================================

_RUN_CACHE: Dict[Tuple, Dict] = {}

def run_model_point(
    T: float,
    burnup: float,
    cand: ManualCase,
    dt_h: float,
    n_modes: int,
    keep_history: bool = False,
):
    key = (
        round(float(T), 6), round(float(burnup), 6),
        cand.label, cand.f_n, cand.K_d, cand.rho_d, cand.fission_rate,
        cand.Dv_scale, cand.Dg_scale, cand.b_scale, cand.gb_scale, cand.gd_scale,
        cand.coalescence_d_scale, cand.capture_scale, cand.D2_xe_scale,
        cand.Dg_D1_scale, cand.Dg_D3_scale, cand.Dv_D1_scale, cand.Dv_D2_scale,
        cand.b_bulk_scale, cand.b_dislocation_scale,
        cand.gd_bubble_scale, cand.gd_line_scale, cand.gd_line_alpha,
        MODEL_FAMILY, USE_PHI_GAS_RESOLUTION, USE_NUCLEATION_MASS_COUPLING, USE_BULK_DISLOCATION_CAPTURE,
        float(dt_h), int(n_modes), bool(keep_history),
    )
    if key in _RUN_CACHE:
        return _RUN_CACHE[key]

    p = UNParameters(
        temperature=float(T),
        fission_rate=cand.fission_rate,
        grain_radius=GRAIN_RADIUS,
        target_burnup_percent_fima=float(burnup),
        dt=float(dt_h) * 3600.0,
        n_modes=int(n_modes),
        xe_yield=XE_YIELD,
        f_n=cand.f_n,
        K_d=cand.K_d,
        rho_d=cand.rho_d,
        Dv_scale=cand.Dv_scale,
        Dg_extra_scale=1.0,
        Dg_scale=cand.Dg_scale,
        D2_xe_scale=cand.D2_xe_scale,
        Dg_D1_scale=cand.Dg_D1_scale,
        Dg_D3_scale=cand.Dg_D3_scale,
        Dv_D1_scale=cand.Dv_D1_scale,
        Dv_D2_scale=cand.Dv_D2_scale,
        b_scale=cand.b_scale,
        b_bulk_scale=cand.b_bulk_scale,
        b_dislocation_scale=cand.b_dislocation_scale,
        gb_scale=cand.gb_scale,
        gd_scale=cand.gd_scale,
        gd_bubble_scale=cand.gd_bubble_scale,
        gd_line_scale=cand.gd_line_scale,
        gd_line_alpha=cand.gd_line_alpha,
        coalescence_d_scale=cand.coalescence_d_scale,
        capture_scale=cand.capture_scale,
        bulk_seed_radius_nm=0.0,
    )

    hist, rates = solve_UN_M7(p, keep_history=keep_history)

    pb_eq = hist["p_b_eq"][-1]
    pd_eq = hist["p_d_eq"][-1]

    row = {
        "T": float(T),
        "burnup": float(burnup),
        "swelling_b_percent": 100.0 * hist["swelling_b"][-1],
        "swelling_d_percent": 100.0 * hist["swelling_d"][-1],
        "swelling_ig_percent": 100.0 * hist["swelling_ig"][-1],
        "Nb": hist["Nb"][-1],
        "Nd": hist["Nd"][-1],
        "Rb_nm": hist["Rb"][-1] * 1.0e9,
        "Rd_nm": hist["Rd"][-1] * 1.0e9,
        "p_b": hist["p_b"][-1],
        "p_b_eq": pb_eq,
        "p_d": hist["p_d"][-1],
        "p_d_eq": pd_eq,
        "p_b_over_eq": hist["p_b"][-1] / pb_eq if pb_eq > 0.0 else math.nan,
        "p_d_over_eq": hist["p_d"][-1] / pd_eq if pd_eq > 0.0 else math.nan,
        "matrix_gas_percent": hist["matrix_gas_percent"][-1],
        "bulk_gas_percent": hist["bulk_gas_percent"][-1],
        "dislocation_gas_percent": hist["dislocation_gas_percent"][-1],
        "qgb_gas_percent": hist["qgb_gas_percent"][-1],
        "f_cap_step_final": hist["f_cap_step"][-1],
        "cap_raw_step_final": hist["cap_raw_step"][-1],
        "capture_fraction_sum": hist["capture_fraction_sum"][-1],
        "capture_raw_sum": hist["capture_raw_sum"][-1],
        "capture_bubbles_cumulative": hist["capture_bubbles_cumulative"][-1],
        "max_f_cap_step": hist["max_f_cap_step"][-1],
        "hist": hist if keep_history else None,
        "rates": rates,
    }

    _RUN_CACHE[key] = row
    return row

## Manual runner and plots

Le celle sotto usano i parametri della prima cella e salvano CSV/PNG.

In [3]:

# ============================================================
# MANUAL RUNNER AND PLOTTING
# ============================================================

import os
import csv
from pathlib import Path
from dataclasses import replace

# ---- Apply manual switches ----

def apply_manual_switches():
    global USE_PHI_GAS_RESOLUTION, USE_NUCLEATION_MASS_COUPLING, USE_BULK_DISLOCATION_CAPTURE
    USE_PHI_GAS_RESOLUTION = bool(MANUAL_USE_PHI_GAS_RESOLUTION)
    USE_NUCLEATION_MASS_COUPLING = bool(MANUAL_USE_NUCLEATION_MASS_COUPLING)
    USE_BULK_DISLOCATION_CAPTURE = bool(MANUAL_USE_BULK_DISLOCATION_CAPTURE)


def make_manual_candidate() -> ManualCase:
    apply_manual_switches()
    p = dict(MANUAL_PARAMS)
    return ManualCase(label=MANUAL_LABEL, **p)


# ---- Manual vacancy diffusivity law ----

def vacancy_diffusivity_UN(p: UNParameters):
    T = p.temperature
    kBT = p.kB_eV * T
    D1 = p.D10_vU * math.exp(-p.Q1_vU / kBT)

    if MANUAL_DV_FORMULA == "v14_fig4_refit":
        A20 = MANUAL_A20_VU_FIG4_REFIT
        D2 = math.sqrt(p.fission_rate) * A20 * math.exp(
            MANUAL_B21_VU / kBT + MANUAL_B22_VU / (kBT**2)
        )
    elif MANUAL_DV_FORMULA == "v14_solver_signs_manual_A20":
        A20 = MANUAL_A20_VU
        D2 = math.sqrt(p.fission_rate) * A20 * math.exp(
            MANUAL_B21_VU / kBT + MANUAL_B22_VU / (kBT**2)
        )
    elif MANUAL_DV_FORMULA == "rizk2025_literal":
        A20 = MANUAL_A20_VU
        expo = -MANUAL_B21_VU / kBT - MANUAL_B22_VU / (kBT**2)
        expo = max(min(expo, 700.0), -745.0)
        D2 = math.sqrt(p.fission_rate) * A20 * math.exp(expo)
    else:
        raise ValueError(f"Unknown MANUAL_DV_FORMULA={MANUAL_DV_FORMULA!r}")

    Dv_unscaled = p.Dv_D1_scale * D1 + p.Dv_D2_scale * D2
    Dv = Dv_unscaled * p.Dv_scale
    return Dv, {
        "Dv1": D1,
        "Dv2": D2,
        "Dv1_scaled": p.Dv_D1_scale * D1,
        "Dv2_scaled": p.Dv_D2_scale * D2,
        "Dv": Dv,
        "Dv_formula": MANUAL_DV_FORMULA,
        "A20_vU_active": A20,
    }


# ---- Optional v14-style rhoSat wrapper, kept outside the core solver ----
RHO_FAB_DEFAULT = 3.0e13
C1_RB = 1.6e14
F0_RB = 2.4
TREF_RB = 1025.0
RHO_SAT_RHO940 = 6.3571
RHO_SAT_RHOINF = 9.1036
RHO_SAT_TAU_K = 203.76


def rho_sat_shape10(T: float) -> float:
    return RHO_SAT_RHOINF - (RHO_SAT_RHOINF - RHO_SAT_RHO940) * math.exp(-(float(T) - 940.0) / RHO_SAT_TAU_K)


def rho_sat_factor_raw(T: float) -> float:
    return rho_sat_shape10(float(T)) / rho_sat_shape10(TREF_RB)


def rho_burnup_1025(F_a_o: float) -> float:
    rho_bu = C1_RB * max(float(F_a_o) - F0_RB, 0.0)
    return max(MANUAL_RHO_FAB, rho_bu)


def effective_rho_d(T: float, burnup: float, cand: ManualCase) -> float:
    if MANUAL_RHO_MODE == "constant":
        return cand.rho_d
    if MANUAL_RHO_MODE == "rhoSat_RayBlank":
        return max(rho_burnup_1025(burnup) * MANUAL_RHO_SCALE * rho_sat_factor_raw(T), 1.0e10)
    raise ValueError(f"Unknown MANUAL_RHO_MODE={MANUAL_RHO_MODE!r}")


def run_model_point_manual(T: float, burnup: float, cand: ManualCase, dt_h: float, n_modes: int, keep_history: bool = False):
    rho_eff = effective_rho_d(T, burnup, cand)
    cand_eff = replace(cand, rho_d=rho_eff, label=f"{cand.label}_rhoEff") if abs(rho_eff - cand.rho_d) > 0.0 else cand
    out = run_model_point(T, burnup, cand_eff, dt_h, n_modes, keep_history=keep_history)
    out["rho_d_eff"] = rho_eff
    return out


# ---- Data utilities ----

def ensure_output_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)


def manual_temperature_grid(tmin=None, tmax=None, step=None):
    tmin = MANUAL_T_MIN if tmin is None else float(tmin)
    tmax = MANUAL_T_MAX if tmax is None else float(tmax)
    step = MANUAL_T_STEP if step is None else float(step)
    n = int(round((tmax - tmin) / step))
    return [tmin + i * step for i in range(n + 1)]


def simulate_manual_grid(cand: ManualCase, dt_h: float, n_modes: int):
    if MANUAL_CLEAR_CACHE_BEFORE_RUN:
        _RUN_CACHE.clear()
    rows = []
    Ts = manual_temperature_grid()
    for bu in MANUAL_BURNUPS:
        print(f"Running burnup {bu:g}% FIMA: {len(Ts)} temperature points")
        for T in Ts:
            out = run_model_point_manual(T, bu, cand, dt_h, n_modes, keep_history=False)
            rates = out.get("rates", {}) or {}
            flat = {k: v for k, v in out.items() if k not in ("hist", "rates")}
            flat.update({
                "label": cand.label,
                "rho_d_eff": out.get("rho_d_eff", cand.rho_d),
                "Dg": rates.get("Dg", math.nan),
                "Dv": rates.get("Dv", math.nan),
                "Dv1": rates.get("Dv1", math.nan),
                "Dv2": rates.get("Dv2", math.nan),
                "A20_vU_active": rates.get("A20_vU_active", math.nan),
                "g_b": rates.get("g_b", math.nan),
                "g_d": rates.get("g_d", math.nan),
                "b_b": rates.get("b_b", math.nan),
                "b_d": rates.get("b_d", math.nan),
                "term_bubbles": rates.get("term_bubbles", math.nan),
                "term_dislocation": rates.get("term_dislocation", math.nan),
                # Geometry diagnostics: if psi >= 1 or porosity becomes too large,
                # the high-T single-size result should be treated as diagnostic only.
                "psi_b": (flat.get("Rb_nm", math.nan) * 1.0e-9 / wigner_seitz_delta(flat.get("Nb", 0.0))) if flat.get("Nb", 0.0) > 0.0 and math.isfinite(flat.get("Rb_nm", math.nan)) else math.nan,
                "psi_d": (flat.get("Rd_nm", math.nan) * 1.0e-9 / wigner_seitz_delta(flat.get("Nd", 0.0))) if flat.get("Nd", 0.0) > 0.0 and math.isfinite(flat.get("Rd_nm", math.nan)) else math.nan,
                "porosity_b": flat.get("swelling_b_percent", math.nan) / 100.0 if math.isfinite(flat.get("swelling_b_percent", math.nan)) else math.nan,
                "porosity_d": flat.get("swelling_d_percent", math.nan) / 100.0 if math.isfinite(flat.get("swelling_d_percent", math.nan)) else math.nan,
            })
            rows.append(flat)
    return rows


def write_manual_csv(rows, filename: str):
    ensure_output_dir(MANUAL_OUTPUT_DIR)
    path = Path(MANUAL_OUTPUT_DIR) / filename
    if not rows:
        return path
    preferred = [
        "label", "burnup", "T", "rho_d_eff", "swelling_b_percent", "swelling_d_percent", "swelling_ig_percent",
        "Rb_nm", "Rd_nm", "Nb", "Nd", "p_b", "p_b_eq", "p_d", "p_d_eq",
        "p_b_over_eq", "p_d_over_eq", "psi_b", "psi_d", "porosity_b", "porosity_d", "matrix_gas_percent", "bulk_gas_percent", "dislocation_gas_percent", "qgb_gas_percent",
        "Dg", "Dv", "Dv1", "Dv2", "A20_vU_active", "g_b", "g_d", "b_b", "b_d",
        "max_f_cap_step", "capture_fraction_sum", "capture_raw_sum", "capture_bubbles_cumulative",
    ]
    keys = []
    for k in preferred:
        if any(k in r for r in rows):
            keys.append(k)
    for r in rows:
        for k in r:
            if k not in keys:
                keys.append(k)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    return path


def rows_for_burnup(rows, bu: float):
    return sorted([r for r in rows if abs(float(r["burnup"]) - float(bu)) < 1e-9], key=lambda r: r["T"])


def row_near(rows, bu: float, T: float):
    sub = rows_for_burnup(rows, bu)
    return min(sub, key=lambda r: abs(float(r["T"]) - float(T)))


def savefig(name: str):
    ensure_output_dir(MANUAL_OUTPUT_DIR)
    path = Path(MANUAL_OUTPUT_DIR) / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    if not MANUAL_SHOW_PLOTS:
        plt.close()
    return path


# ---- Plot helpers, matching capture_only diagnostic set ----

def plot_exp_swelling_for_burnup(bu: float):
    series_names = sorted({p["series"] for p in EXP_SWELLING_T if abs(p["burnup"] - bu) < 1.0e-9})
    for series in series_names:
        pts = [p for p in EXP_SWELLING_T if abs(p["burnup"] - bu) < 1.0e-9 and p["series"] == series]
        marker = "x" if "119" in series else "^"
        plt.scatter([p["T"] for p in pts], [p["swelling"] for p in pts], marker=marker, s=65, label=f"Exp P2 {series}", zorder=5)


def plot_swelling_T(rows, bu: float, prefix: str):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.plot(Ts, [r["swelling_d_percent"] for r in sub], label="M7 dislocation/P2 swelling")
    plt.plot(Ts, [r["swelling_b_percent"] for r in sub], linestyle="--", label="M7 bulk swelling")
    plot_exp_swelling_for_burnup(bu)
    plt.xlabel("T [K]")
    plt.ylabel("Fission gas swelling [%]")
    plt.title(f"{prefix}: swelling vs T at {bu:.1f}% FIMA")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_swelling_T_{bu:.1f}FIMA.png")


def plot_swelling_burnup_1600(cand: ManualCase, prefix: str):
    burnup_grid = [0.2, 0.5, 0.8, 1.1, 1.3, 1.6, 2.0, 2.5, 3.2, 4.0, 5.0, 6.0]
    rows_bu = [run_model_point_manual(1600.0, bu, cand, MANUAL_DT_H, MANUAL_N_MODES, keep_history=False) for bu in burnup_grid]
    plt.figure(figsize=(9, 5.8))
    plt.plot(burnup_grid, [r["swelling_d_percent"] for r in rows_bu], label="M7 dislocation/P2 swelling")
    plt.plot(burnup_grid, [r["swelling_b_percent"] for r in rows_bu], linestyle="--", label="M7 bulk swelling")
    plt.scatter([p["burnup"] for p in EXP_SWELLING_BURNUP_1600], [p["swelling"] for p in EXP_SWELLING_BURNUP_1600], marker="^", s=80, label="Exp P2 approx")
    plt.xlabel("Burnup [% FIMA]")
    plt.ylabel("Fission gas swelling [%]")
    plt.title(f"{prefix}: swelling vs burnup at 1600 K")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_swelling_vs_burnup_1600K.png")


def plot_radius_T(rows, bu: float, prefix: str):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.plot(Ts, [r["Rd_nm"] for r in sub], label="M7 dislocation R_d")
    plt.plot(Ts, [r["Rb_nm"] for r in sub], linestyle="--", label="M7 bulk R_b")
    if abs(bu - 1.3) < 1e-9:
        plt.scatter([p["T"] for p in EXP_RD_T_13], [p["R_nm"] for p in EXP_RD_T_13], marker="^", s=70, label="Exp P2 / large bubbles")
    plt.yscale("log")
    plt.xlabel("T [K]")
    plt.ylabel("Bubble radius [nm]")
    plt.title(f"{prefix}: bubble radius at {bu:.1f}% FIMA")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_radius_T_{bu:.1f}FIMA.png")


def plot_concentration_T(rows, bu: float, prefix: str):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.plot(Ts, [r["Nd"] for r in sub], label="M7 dislocation N_d")
    plt.plot(Ts, [r["Nb"] for r in sub], linestyle="--", label="M7 bulk N_b")
    if abs(bu - 1.3) < 1e-9:
        plt.scatter([p["T"] for p in EXP_ND_T_13], [p["N"] for p in EXP_ND_T_13], marker="^", s=70, label="Exp P2 / large bubbles")
    plt.yscale("log")
    plt.xlabel("T [K]")
    plt.ylabel("Bubble concentration [m$^{-3}$]")
    plt.title(f"{prefix}: bubble concentration at {bu:.1f}% FIMA")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_concentration_T_{bu:.1f}FIMA.png")


def plot_pressure_T(rows, bu: float, prefix: str):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.plot(Ts, [r["p_b"] for r in sub], label="Bulk pressure p_b")
    plt.plot(Ts, [r["p_b_eq"] for r in sub], linestyle="--", label="Bulk equilibrium p_b,eq")
    plt.plot(Ts, [r["p_d"] for r in sub], label="Dislocation pressure p_d")
    plt.plot(Ts, [r["p_d_eq"] for r in sub], linestyle="--", label="Dislocation equilibrium p_d,eq")
    plt.yscale("log")
    plt.xlabel("T [K]")
    plt.ylabel("Pressure [Pa]")
    plt.title(f"{prefix}: pressure diagnostic at {bu:.1f}% FIMA")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_pressure_T_{bu:.1f}FIMA.png")


def plot_gas_partition(rows, bu: float, prefix: str):
    # extend to MANUAL_GAS_PARTITION_T_MAX if needed
    cand = make_manual_candidate()
    Ts = manual_temperature_grid(MANUAL_T_MIN, MANUAL_GAS_PARTITION_T_MAX, 100.0)
    rows_gas = [run_model_point_manual(T, bu, cand, MANUAL_DT_H, MANUAL_N_MODES, keep_history=False) for T in Ts]
    plt.figure(figsize=(9, 5.8))
    plt.plot(Ts, [r["matrix_gas_percent"] for r in rows_gas], label="Matrix")
    plt.plot(Ts, [r["bulk_gas_percent"] for r in rows_gas], linestyle="--", label="Bulk bubbles")
    plt.plot(Ts, [r["dislocation_gas_percent"] for r in rows_gas], linestyle="-.", label="Dislocation bubbles")
    plt.plot(Ts, [r["qgb_gas_percent"] for r in rows_gas], linestyle=":", label="Gas to grain face q_gb")
    plt.xlabel("T [K]")
    plt.ylabel("Amount of generated gas [%]")
    plt.title(f"{prefix}: gas partition at {bu:.1f}% FIMA")
    plt.xlim(MANUAL_T_MIN, MANUAL_GAS_PARTITION_T_MAX)
    plt.ylim(0, 100)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_gas_partition_{bu:.1f}FIMA.png")


def plot_capture_diagnostic(rows, bu: float, prefix: str):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.plot(Ts, [r["max_f_cap_step"] for r in sub], label="max f_cap step, clipped fraction")
    plt.plot(Ts, [r["capture_fraction_sum"] for r in sub], label="sum clipped f_cap steps, diagnostic")
    plt.plot(Ts, [r["capture_raw_sum"] for r in sub], label="sum raw capture hazard, diagnostic")
    plt.yscale("symlog", linthresh=1e-5)
    plt.xlabel("T [K]")
    plt.ylabel("Capture diagnostic")
    plt.title(f"{prefix}: capture diagnostics at {bu:.1f}% FIMA")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_capture_diagnostic_{bu:.1f}FIMA.png")



def plot_diffusivities(rows, prefix: str):
    """Plot Dg, Dv, Dv1, Dv2 vs temperature.
    For fixed fission rate these curves are the same for all burnups, so we use the first burnup.
    """
    bu = MANUAL_BURNUPS[0]
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.semilogy(Ts, [max(r["Dg"], 1e-300) for r in sub], label="Dg = D1_Xe + D3_Xe")
    plt.semilogy(Ts, [max(r["Dv"], 1e-300) for r in sub], label="Dv total")
    plt.semilogy(Ts, [max(r["Dv1"], 1e-300) for r in sub], linestyle="--", label="Dv1 thermal")
    plt.semilogy(Ts, [max(r["Dv2"], 1e-300) for r in sub], linestyle=":", label="Dv2 irradiation")
    plt.xlabel("T [K]")
    plt.ylabel("Diffusivity [m$^2$/s]")
    plt.title(f"{prefix}: diffusivities vs T")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{prefix}_diffusivities_T.png")

def make_manual_plots(rows, cand: ManualCase):
    prefix = cand.label
    saved = []

    # Same main plot list as capture_only diagnostics, except no pressure-ratio plot.
    for bu in MANUAL_BURNUPS:
        saved.append(plot_swelling_T(rows, bu, prefix))
    saved.append(plot_swelling_burnup_1600(cand, prefix))
    saved.append(plot_radius_T(rows, 1.3, prefix))
    saved.append(plot_concentration_T(rows, 1.3, prefix))
    saved.append(plot_pressure_T(rows, 3.2, prefix))
    saved.append(plot_gas_partition(rows, 1.1, prefix))
    saved.append(plot_gas_partition(rows, 3.2, prefix))
    saved.append(plot_capture_diagnostic(rows, 3.2, prefix))
    saved.append(plot_diffusivities(rows, prefix))

    if PLOT_EXTRA_ALL_BURNUPS:
        for bu in MANUAL_BURNUPS:
            if abs(bu - 1.3) > 1e-9:
                saved.append(plot_radius_T(rows, bu, prefix))
                saved.append(plot_concentration_T(rows, bu, prefix))
            if abs(bu - 3.2) > 1e-9:
                saved.append(plot_pressure_T(rows, bu, prefix))
            if abs(bu - 1.1) > 1e-9 and abs(bu - 3.2) > 1e-9:
                saved.append(plot_gas_partition(rows, bu, prefix))
                saved.append(plot_capture_diagnostic(rows, bu, prefix))

    return saved


def print_manual_summary(rows, cand: ManualCase):
    print("\n" + "=" * 120)
    print(f"Manual case: {cand.label}")
    print("=" * 120)
    print("Family switches:")
    print(f"  phi in gas resolution       = {USE_PHI_GAS_RESOLUTION}")
    print(f"  nucleation mass coupling    = {USE_NUCLEATION_MASS_COUPLING}")
    print(f"  bulk→dislocation capture    = {USE_BULK_DISLOCATION_CAPTURE}")
    print(f"  vacancy Dv formula          = {MANUAL_DV_FORMULA}")
    print(f"  rho_d mode                  = {MANUAL_RHO_MODE}")
    print("\nManualCase parameters:")
    for k, v in MANUAL_PARAMS.items():
        print(f"  {k:26s} = {v:.6g}" if isinstance(v, (int, float)) else f"  {k:26s} = {v}")

    print("\nSelected diagnostics:")
    header = f"{'bu [%]':>7s} {'T [K]':>7s} {'sw_d [%]':>10s} {'R_d [nm]':>10s} {'N_d [m^-3]':>13s} {'p_d [Pa]':>12s} {'p_d_eq [Pa]':>12s} {'qgb [%]':>9s}"
    print(header)
    print("-" * len(header))
    for bu in MANUAL_BURNUPS:
        for T in [1200.0, 1600.0, 1800.0, 2000.0]:
            if T < MANUAL_T_MIN or T > MANUAL_T_MAX:
                continue
            r = row_near(rows, bu, T)
            print(f"{bu:7.2f} {r['T']:7.0f} {r['swelling_d_percent']:10.3g} {r['Rd_nm']:10.3g} {r['Nd']:13.3e} {r['p_d']:12.3e} {r['p_d_eq']:12.3e} {r['qgb_gas_percent']:9.3g}")
    print("=" * 120)


def main_manual():
    ensure_output_dir(MANUAL_OUTPUT_DIR)
    cand = make_manual_candidate()
    rows = simulate_manual_grid(cand, MANUAL_DT_H, MANUAL_N_MODES)
    csv_path = write_manual_csv(rows, f"{cand.label}_manual_grid.csv")
    print_manual_summary(rows, cand)
    saved = make_manual_plots(rows, cand)
    print(f"\nCSV written to: {csv_path}")
    print("Plots written:")
    for p in saved:
        print(f"  - {p}")
    return rows


In [4]:
# Run the current manual configuration
rows = main_manual()

Running burnup 1.1% FIMA: 35 temperature points
Running burnup 1.3% FIMA: 35 temperature points
Running burnup 3.2% FIMA: 35 temperature points

Manual case: manual_Rizk2025_capture_literalDv
Family switches:
  phi in gas resolution       = False
  nucleation mass coupling    = False
  bulk→dislocation capture    = True
  vacancy Dv formula          = rizk2025_literal
  rho_d mode                  = constant

ManualCase parameters:
  f_n                        = 1e-06
  K_d                        = 500000
  rho_d                      = 3e+13
  fission_rate               = 5e+19
  Dv_scale                   = 1
  Dg_scale                   = 1
  b_scale                    = 1
  gb_scale                   = 1
  gd_scale                   = 1
  coalescence_d_scale        = 1
  capture_scale              = 1
  Dg_D1_scale                = 1
  Dg_D3_scale                = 1
  Dv_D1_scale                = 1
  Dv_D2_scale                = 1
  b_bulk_scale               = 1
  b_dislocation_sca

In [5]:
# Optional: inspect a single point with full history using the same manual configuration.
# Change T_INSPECT and BU_INSPECT, then run this cell.
T_INSPECT = 1600.0
BU_INSPECT = 1.3
cand = make_manual_candidate()
out = run_model_point_manual(T_INSPECT, BU_INSPECT, cand, MANUAL_DT_H, MANUAL_N_MODES, keep_history=True)
print({k: out[k] for k in ["T", "burnup", "swelling_d_percent", "swelling_b_percent", "Rd_nm", "Nd", "p_d", "p_d_eq", "qgb_gas_percent", "rho_d_eff"]})
print("rates:")
for k in ["Dg", "Dv", "Dv1", "Dv2", "A20_vU_active", "g_b", "g_d", "b_b", "b_d", "term_bubbles", "term_dislocation"]:
    print(f"  {k:20s} = {out['rates'].get(k)}")


{'T': 1600.0, 'burnup': 1.3, 'swelling_d_percent': 0.2501021112900127, 'swelling_b_percent': 1.0814537516126916, 'Rd_nm': 34.25820786136873, 'Nd': 1.485028635979416e+19, 'p_d': 64802000.31168896, 'p_d_eq': 64801988.73751896, 'qgb_gas_percent': 16.316100355600263, 'rho_d_eff': 30000000000000.0}
rates:
  Dg                   = 5.218480065953793e-19
  Dv                   = 6.867340206203763e-07
  Dv1                  = 2.005018746656425e-20
  Dv2                  = 6.867340206203563e-07
  A20_vU_active        = 1.32e-19
  g_b                  = 0.0008404920394692911
  g_d                  = 3.061183933993112e-05
  b_b                  = 7.336580796893374e-06
  b_d                  = 3.836553305878453e-06
  term_bubbles         = 3.3566600320971088e-06
  term_dislocation     = 2.7255179307834008e-05
